# Inspeção de dados CFPB

## Contexto do negócio

O [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/) é a base de reclamações de consumidores publicada pelo Consumer Financial Protection Bureau (CFPB). Cada registro preserva as opções de `Product` e `Issue` escolhidas pelo consumidor, conforme o formulário vigente na data do envio. Parte das reclamações traz também uma narrativa em texto livre: 3.836.659 das 17.094.898 linhas do snapshot usado (22,4%).

A reclamação passa por algumas etapas, descritas pelo próprio CFPB no [processo de reclamação](https://www.consumerfinance.gov/complaint/process/). O consumidor a envia pelo site ou por telefone, ou outra agência governamental a encaminha ao CFPB, que a repassa à empresa citada; quando outra agência está mais apta a ajudar, a reclamação é enviada a ela. A empresa em geral responde em 15 dias e, em alguns casos, dá a resposta final em até 60 dias, e o consumidor tem 60 dias para avaliar essa resposta. A reclamação é publicada na base depois da resposta da empresa ou após 15 dias, o que ocorrer primeiro, sem nomes, contatos, números de conta ou outros dados que identifiquem diretamente o consumidor ([como o CFPB compartilha os dados](https://www.consumerfinance.gov/complaint/data-use/)). Em 14 de agosto de 2026, o CFPB [deixou de publicar as narrativas](https://www.consumerfinance.gov/about-us/newsroom/the-cfpb-to-cease-discretionary-publication-of-complaint-narratives-and-visualizations/) e as transferiu para sua sala de leitura FOIA, o que limita a reconstrução exata do snapshot de narrativas usado aqui.

Rotear uma narrativa para a família de produto correta é a tarefa operacional do projeto, que trabalha com nove famílias. A classe crítica, `debt_credit_management`, soma 5.437 linhas em 3,8 milhões de narrativas (0,14%). É nela que a falha de roteamento é cara e que toda abordagem clássica do projeto falhou.

Duas leituras são recusadas. O volume de reclamações não mede prevalência de mercado nem dano ao consumidor, porque o alvo é o roteamento para categorias históricas do formulário. E `Issue` não é um rótulo global estável: a [referência de campos](https://cfpb.github.io/api/ccdb/fields.html) declara que seus valores possíveis dependem de `Product`, 178 rótulos brutos colidem entre famílias, e as opções do formulário mudaram com o tempo, como nas [opções de produto e issue de agosto de 2023](https://files.consumerfinance.gov/f/documents/cfpb_consumer_complaint_form_product_issue_options_August_2023_FINAL.pdf).

Este notebook faz a inspeção inicial dos dados que sustentam essas escolhas.

Entrada fina para o perfil inicial. O notebook 01 executado na raiz permanece como registro histórico; esta versão chama `src`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'dataset' / 'processed' / 'complaints.parquet').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find project root')
    PROJECT_ROOT = parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from consumer_complaint_intelligence.config import ProjectPaths
from consumer_complaint_intelligence.data import read_parquet_sample

paths = ProjectPaths.from_root(PROJECT_ROOT)
columns = [
    'Date received',
    'Product',
    'Issue',
    'Consumer complaint narrative',
]
sample = read_parquet_sample(
    paths.require_parquet(), limit=25_000, columns=columns
)
sample.head(10)


A amostra é limitada a 25.000 linhas e não representa o corpus. Decisões de taxonomia e split pertencem à S0.
